In [105]:
!pip install pennylane

In [106]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pennylane as qml
from pennylane import numpy as np

In [107]:
# import iris dataset
url = 'https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/0e7a9b0a5d22642a06d3d5b9bcbad9890c8ee534/iris.csv'
my_df = pd.read_csv(url)
my_df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [108]:
# drop virginica samples
my_df = my_df.drop(my_df[my_df['species'] == 'virginica'].index)
my_df['species']

,species
0,setosa
1,setosa
2,setosa
3,setosa
4,setosa
...,...
95,versicolor
96,versicolor
97,versicolor
98,versicolor


In [109]:
# define labels (-1 for setosa and 1 for versicolor)
my_df['species'] = my_df['species'].replace('setosa', -1)
my_df['species'] = my_df['species'].replace('versicolor', 1)
my_df['species']

/tmp/ipykernel_2283/1501611613.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  my_df['species'] = my_df['species'].replace('versicolor', 1)


,species
0,-1
1,-1
2,-1
3,-1
4,-1
...,...
95,1
96,1
97,1
98,1


In [110]:
X = my_df.drop('species', axis = 1)
y = my_df['species']

In [111]:
# convert to numpy arrays
X = X.values
y = y. values

In [112]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [113]:
# scaling
scaler = MinMaxScaler(feature_range=(0, 2*np.pi))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [114]:
# create quantum device

n_qubits = 4
np.random.seed(42)
trainable_weights = np.random.uniform(-0.1, 0.1, size=n_qubits)

dev = qml.device("default.qubit", wires = n_qubits)


@qml.qnode(dev)
def qc(x, weights):
  for i in range(n_qubits):
     qml.RY(x[i], wires = i)
  for i in range(n_qubits):
     qml.RY(weights[i], wires = i)
  for i in range(n_qubits - 1):
    qml.CNOT(wires=[i, i + 1])
  return qml.expval(qml.PauliZ(0))

print(qml.draw(qc)(X_train[0],trainable_weights))





0: ──RY(3.26)──RY(-0.03)─╭●───────┤  <Z>
1: ──RY(2.09)──RY(0.09)──╰X─╭●────┤     
2: ──RY(5.62)──RY(0.05)─────╰X─╭●─┤     
3: ──RY(5.03)──RY(0.02)────────╰X─┤     


In [115]:
# define mse loss function
def mse_loss(weights, features, pred_labels):
    loss = 0.0
    for x, y_true in zip(features, pred_labels):
        y_pred = qc(x, weights)
        loss += (y_true - y_pred) ** 2
    return loss / len(features)

In [116]:
# optimizing through gradient descent

opt = qml.GradientDescentOptimizer(stepsize=0.5)

epochs = 20


for epoch in range(epochs):

    trainable_weights, current_loss = opt.step_and_cost(
        mse_loss,
        trainable_weights,
        features=X_train,
        pred_labels=y_train
    )

    print(f"Epoch {epoch+1} | Average Loss: {current_loss:.4f}")

print("\nTraining complete!")


Epoch 1 | Average Loss: 1.8873
Epoch 2 | Average Loss: 1.4129
Epoch 3 | Average Loss: 0.9720
Epoch 4 | Average Loss: 0.6941
Epoch 5 | Average Loss: 0.5646
Epoch 6 | Average Loss: 0.5154
Epoch 7 | Average Loss: 0.4989
Epoch 8 | Average Loss: 0.4937
Epoch 9 | Average Loss: 0.4921
Epoch 10 | Average Loss: 0.4916
Epoch 11 | Average Loss: 0.4915
Epoch 12 | Average Loss: 0.4914
Epoch 13 | Average Loss: 0.4914
Epoch 14 | Average Loss: 0.4914
Epoch 15 | Average Loss: 0.4914
Epoch 16 | Average Loss: 0.4914
Epoch 17 | Average Loss: 0.4914
Epoch 18 | Average Loss: 0.4914
Epoch 19 | Average Loss: 0.4914
Epoch 20 | Average Loss: 0.4914

Training complete!


In [117]:
# evaluating on test data


correct_predictions = 0

print("Testing the model on test set...\n")

for x_sample, y_true in zip(X_test, y_test):
    raw_prediction = qc(x_sample, trainable_weights)


    if raw_prediction >= 0:
        final_guess = 1
    else:
        final_guess = -1

    if final_guess == y_true:
        correct_predictions += 1

    true_name = "setosa" if y_true == 1 else "versicolor"
    guess_name = "setosa" if final_guess == 1 else "versicolor"
    print(f"True Species: {true_name:<10} | Model Guess: {guess_name:<10} | Raw Score: {raw_prediction:.4f}")

accuracy = (correct_predictions / len(X_test)) * 100
print(f"\nFinal Test Accuracy: {accuracy:.1f}%")

Testing the model on test set...

True Species: setosa     | Model Guess: setosa     | Raw Score: 0.9575
True Species: setosa     | Model Guess: setosa     | Raw Score: 0.1143
True Species: setosa     | Model Guess: setosa     | Raw Score: 0.8651
True Species: versicolor | Model Guess: versicolor | Raw Score: -0.9984
True Species: versicolor | Model Guess: versicolor | Raw Score: -0.7286
True Species: versicolor | Model Guess: versicolor | Raw Score: -0.7286
True Species: versicolor | Model Guess: versicolor | Raw Score: -0.9175
True Species: setosa     | Model Guess: setosa     | Raw Score: 0.1143
True Species: versicolor | Model Guess: versicolor | Raw Score: -0.1179
True Species: versicolor | Model Guess: versicolor | Raw Score: -0.7286
True Species: versicolor | Model Guess: setosa     | Raw Score: 0.5480
True Species: versicolor | Model Guess: versicolor | Raw Score: -0.9984
True Species: setosa     | Model Guess: setosa     | Raw Score: 0.9982
True Species: versicolor | Model Gue